# 5. Exploratory Data Analysis with SQL
SQLite allows SQL queries to be run directly on a pandas DataFrame.

In [1]:
import pandas as pd
import sqlite3
from IPython.display import display

df = pd.read_csv("https://raw.githubusercontent.com/chuksoo/IBM-Data-Science-Capstone-SpaceX/main/Spacex.csv")
print("Shape:", df.shape)
print(df.columns.tolist())
display(df.head())
conn = sqlite3.connect(":memory:")
df.to_sql("SPACEXTBL", conn, index=False, if_exists="replace")

Shape: (101, 10)
['Date', 'Time (UTC)', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing _Outcome']


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing _Outcome
0,2010-04-06,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-08-12,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-08-10,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-01-03,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


101

In [2]:
# Show the actual table column names to adapt queries if needed
print(pd.read_sql_query("PRAGMA table_info(SPACEXTBL)", conn))

   cid               name     type  notnull dflt_value  pk
0    0               Date     TEXT        0       None   0
1    1         Time (UTC)     TEXT        0       None   0
2    2    Booster_Version     TEXT        0       None   0
3    3        Launch_Site     TEXT        0       None   0
4    4            Payload     TEXT        0       None   0
5    5  PAYLOAD_MASS__KG_  INTEGER        0       None   0
6    6              Orbit     TEXT        0       None   0
7    7           Customer     TEXT        0       None   0
8    8    Mission_Outcome     TEXT        0       None   0
9    9   Landing _Outcome     TEXT        0       None   0


In [3]:
# First 20 rows
display(pd.read_sql_query('SELECT * FROM SPACEXTBL LIMIT 20', conn))

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing _Outcome
0,2010-04-06,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-08-12,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-08-10,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-01-03,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt
5,2013-09-23,16:00:00,F9 v1.1 B1003,VAFB SLC-4E,CASSIOPE,500,Polar LEO,MDA,Success,Uncontrolled (ocean)
6,2013-03-12,22:41:00,F9 v1.1,CCAFS LC-40,SES-8,3170,GTO,SES,Success,No attempt
7,2014-06-01,22:06:00,F9 v1.1,CCAFS LC-40,Thaicom 6,3325,GTO,Thaicom,Success,No attempt
8,2014-04-18,19:25:00,F9 v1.1,CCAFS LC-40,SpaceX CRS-3,2296,LEO (ISS),NASA (CRS),Success,Controlled (ocean)
9,2014-07-14,15:15:00,F9 v1.1,CCAFS LC-40,OG2 Mission 1 6 Orbcomm-OG2 satellites,1316,LEO,Orbcomm,Success,Controlled (ocean)


In [4]:
# Unique launch sites — identify the exact column name first
cols = df.columns.tolist()
launch_col = next((c for c in cols if c.lower().replace("_", "") in ["launchsite"]), None)
print("Launch-site column:", launch_col)
if launch_col:
    q = f'SELECT DISTINCT "{launch_col}" FROM SPACEXTBL'
    display(pd.read_sql_query(q, conn))

Launch-site column: Launch_Site


,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


In [5]:
# Mission outcome counts by launch site (adapt column names if they differ)
outcome_col = next((c for c in df.columns if "outcome" in c.lower()), None)
if launch_col and outcome_col:
    q = f'SELECT "{launch_col}" AS LaunchSite, "{outcome_col}" AS MissionOutcome, COUNT(*) AS Count FROM SPACEXTBL GROUP BY "{launch_col}", "{outcome_col}" ORDER BY LaunchSite'
    display(pd.read_sql_query(q, conn))
else:
    print("Could not automatically identify launch-site/outcome columns. Use the column list above to adjust.")

,LaunchSite,MissionOutcome,Count
0,CCAFS LC-40,Failure (in flight),1
1,CCAFS LC-40,Success,25
2,CCAFS SLC-40,Success,32
3,CCAFS SLC-40,Success,1
4,CCAFS SLC-40,Success (payload status unclear),1
5,KSC LC-39A,Success,25
6,VAFB SLC-4E,Success,16


In [6]:
# Minimum payload mass, if a payload-mass column is present
payload_col = next((c for c in df.columns if "payload" in c.lower() and "mass" in c.lower()), None)
if payload_col:
    q = f'SELECT MIN("{payload_col}") AS MinimumPayloadMass FROM SPACEXTBL'
    display(pd.read_sql_query(q, conn))

,MinimumPayloadMass
0,0


## SQL interpretation
Include the query, result table, and one sentence describing what the result means. Column names can differ between course CSV versions, so this notebook detects common names where possible.